# 🚲 Proyek Analisis Data: Bike Sharing Dataset
- **Nama:** Senior Data Analyst
- **Email:** analyst@dicoding.local
- **ID Dicoding:** dicoding_analyst
- **Dataset:** Bike Sharing Dataset (Capital Bikeshare)


## 1. Menentukan Pertanyaan Bisnis (SMART Framework)

Untuk memandu analisis data, dirumuskan dua pertanyaan bisnis berbasis kaidah **SMART** (*Specific, Measurable, Action-Oriented, Relevant, Time-bound*):

1. **Pertanyaan Bisnis 1 (Dampak Musim & Cuaca):**
   - *Pertanyaan:* **Bagaimana pengaruh kondisi cuaca dan musim terhadap jumlah rata-rata penyewaan sepeda harian selama periode 2011–2012, dan musim apa yang mencatatkan performa penyewaan tertinggi?**
   - *Kriteria SMART:*
     - **Specific:** Fokus mengukur pengaruh variabel lingkungan (cuaca dan musim) terhadap performa rental sepeda.
     - **Measurable:** Diukur dengan metrik kuantitatif rata-rata (*mean*) dan total volume penyewaan (`cnt`).
     - **Action-Oriented:** Menjadi dasar tim operasional dalam menjadwalkan kuota unit sepeda dan masa servis armada.
     - **Relevant:** Berdampak langsung pada kesiapan ketersediaan armada sepeda sesuai iklim.
     - **Time-bound:** Dibatasi pada horizon data 2 tahun (Januari 2011 – Desember 2012).

2. **Pertanyaan Bisnis 2 (Pola Pengguna pada Jam Sibuk):**
   - *Pertanyaan:* **Bagaimana pola perbandingan volume penyewaan antara pengguna casual dan registered berdasarkan jam dalam sehari (khususnya jam sibuk komuter 07:00–09:00 dan 17:00–19:00) pada hari kerja (working day)?**
   - *Kriteria SMART:*
     - **Specific:** Menelaah disparitas perilaku antara pengguna terdaftar (*registered*) dan pengguna insidental (*casual*) pada jam sibuk hari kerja.
     - **Measurable:** Dihitung dari rata-rata volume sewa per jam untuk masing-masing kelompok pengguna.
     - **Action-Oriented:** Mengarahkan tim pemasaran untuk konversi langganan dan tim operasional untuk rebalancing stasiun docking.
     - **Relevant:** Esensial untuk optimasi kapasitas docking station dan pendapatan berulang (*recurring revenue*).
     - **Time-bound:** Mencakup data observasi operasional tahun 2011–2012.


## 2. Menyiapkan Semua Library yang Dibutuhkan


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Atur konfigurasi visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11


## 3. Data Wrangling

### 3.1 Gathering Data
Memuat dataset `day.csv` (data harian) dan `hour.csv` (data per jam).


In [ ]:
# Memuat kedua berkas dataset
day_df = pd.read_csv('data/day.csv')
hour_df = pd.read_csv('data/hour.csv')

print("Dimensi day_df :", day_df.shape)
print("Dimensi hour_df:", hour_df.shape)


In [ ]:
# Menampilkan 5 baris pertama data harian
day_df.head()


In [ ]:
# Menampilkan 5 baris pertama data per jam
hour_df.head()


### 3.2 Assessing Data
Memeriksa struktur, tipe data, missing value, duplikasi, dan sebaran nilai statistika untuk mengidentifikasi anomali/permasalahan pada data.


In [ ]:
# Informasi tipe data dan non-null values
print("=== Informasi day_df ===")
day_df.info()

print()
print("=== Informasi hour_df ===")
hour_df.info()


In [ ]:
# Pengecekan Missing Values
print("Missing value pada day_df:")
print(day_df.isna().sum())

print()
print("Missing value pada hour_df:")
print(hour_df.isna().sum())


In [ ]:
# Pengecekan Duplikasi
print("Jumlah duplikasi data day_df :", day_df.duplicated().sum())
print("Jumlah duplikasi data hour_df:", hour_df.duplicated().sum())


In [ ]:
# Ringkasan Statistika Deskriptif
day_df.describe()


**Temuan Masalah pada Tahap Assessing:**
1. **Tipe Data `dteday`:** Kolom tanggal `dteday` pada kedua dataframe terbaca sebagai tipe objek (`object`/string), padahal seharusnya bertipe `datetime64` agar dapat dioperasikan secara runtun waktu (*time-series*).
2. **Keterbacaan Nilai Kategorik (*Readability*):** Variabel kategorik kunci seperti `season` (1–4), `weathersit` (1–4), `yr` (0–1), `mnth` (1–12), `weekday` (0–6), dan `workingday` (0–1) tersimpan dalam representasi integer angka kode, yang menyulitkan interpretasi cepat saat eksplorasi dan visualisasi.
3. **Representasi Nilai Kontinu Ternormalisasi:** Kolom cuaca (`temp`, `atemp`, `hum`, `windspeed`) tersimpan dalam format ternormalisasi (0 hingga 1), sehingga perlu dibuatkan nilai skala aslinya (°C, %, km/h) untuk keperluan interpretasi bisnis yang intuitif.


### 3.3 Cleaning Data
Melakukan pembersihan data sesuai dengan temuan pada tahap *Assessing*.


In [ ]:
# 1. Konversi tipe data dteday menjadi datetime
day_df['dteday'] = pd.to_datetime(day_df['dteday'])
hour_df['dteday'] = pd.to_datetime(hour_df['dteday'])

# 2. Pemetaan kode integer ke label deskriptif yang mudah dipahami
season_mapping = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
weather_mapping = {
    1: 'Clear/Partly Cloudy',
    2: 'Misty/Cloudy',
    3: 'Light Snow/Rain',
    4: 'Severe Weather'
}
month_mapping = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}
weekday_mapping = {
    0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday',
    4: 'Thursday', 5: 'Friday', 6: 'Saturday'
}
workingday_mapping = {0: 'Weekend/Holiday', 1: 'Working Day'}
year_mapping = {0: 2011, 1: 2012}

# Terapkan pemetaan pada day_df
day_df['season_label'] = day_df['season'].map(season_mapping)
day_df['weather_label'] = day_df['weathersit'].map(weather_mapping)
day_df['month_label'] = day_df['mnth'].map(month_mapping)
day_df['weekday_label'] = day_df['weekday'].map(weekday_mapping)
day_df['workingday_label'] = day_df['workingday'].map(workingday_mapping)
day_df['year_label'] = day_df['yr'].map(year_mapping)

# Terapkan pemetaan pada hour_df
hour_df['season_label'] = hour_df['season'].map(season_mapping)
hour_df['weather_label'] = hour_df['weathersit'].map(weather_mapping)
hour_df['month_label'] = hour_df['mnth'].map(month_mapping)
hour_df['weekday_label'] = hour_df['weekday'].map(weekday_mapping)
hour_df['workingday_label'] = hour_df['workingday'].map(workingday_mapping)
hour_df['year_label'] = hour_df['yr'].map(year_mapping)

# 3. Denormalisasi metrik cuaca ke satuan riil
day_df['temp_celsius'] = (day_df['temp'] * 41).round(2)
day_df['atemp_celsius'] = (day_df['atemp'] * 50).round(2)
day_df['hum_pct'] = (day_df['hum'] * 100).round(2)
day_df['windspeed_kmh'] = (day_df['windspeed'] * 67).round(2)

hour_df['temp_celsius'] = (hour_df['temp'] * 41).round(2)
hour_df['atemp_celsius'] = (hour_df['atemp'] * 50).round(2)
hour_df['hum_pct'] = (hour_df['hum'] * 100).round(2)
hour_df['windspeed_kmh'] = (hour_df['windspeed'] * 67).round(2)

print("Pembersihan data selesai. Tipe dteday:", day_df['dteday'].dtype)


In [ ]:
# Verifikasi hasil pembersihan data
day_df[['dteday', 'season_label', 'weather_label', 'temp_celsius', 'workingday_label', 'cnt']].head()


## 4. Exploratory Data Analysis (EDA)

### 4.1 Eksplorasi Pertanyaan 1: Pengaruh Musim & Cuaca terhadap Penyewaan Sepeda
Kita melakukan agregasi untuk melihat rata-rata, standar deviasi, dan total rental berdasarkan kombinasi musim dan kondisi cuaca.


In [ ]:
# Agregasi penyewaan berdasarkan musim
season_summary = day_df.groupby('season_label')['cnt'].agg(
    total_rentals='sum',
    avg_rentals='mean',
    median_rentals='median',
    std_rentals='std'
).sort_values(by='avg_rentals', ascending=False)

season_summary


*Insight Analisis Musim:*
- **Fall (Musim Gugur)** mencatatkan rata-rata penyewaan harian tertinggi sebesar ~**5.644 unit/hari**, disusul oleh **Summer (Musim Panas)** dengan ~**4.992 unit/hari**.
- **Spring (Musim Semi)** mencatatkan angka terendah (~**2.604 unit/hari**), dikarenakan suhu awal musim semi yang masih cukup dingin serta curah hujan transisi yang tidak menentu.


In [ ]:
# Agregasi penyewaan berdasarkan kondisi cuaca
weather_summary = day_df.groupby('weather_label')['cnt'].agg(
    total_rentals='sum',
    avg_rentals='mean',
    std_rentals='std',
    count_days='count'
).sort_values(by='avg_rentals', ascending=False)

weather_summary


*Insight Analisis Cuaca:*
- Kondisi **Clear/Partly Cloudy** sangat mendominasi dengan rata-rata **4.877 unit/hari** (total lebih dari 2,25 juta rental).
- Ketika terjadi **Light Snow/Rain**, rata-rata sewa merosot drastis menjadi **1.803 unit/hari** (penurunan lebih dari 60%). Pada data harian tidak tercatat hari dengan kategori badai ekstrim (*Severe Weather*).


### 4.2 Eksplorasi Pertanyaan 2: Pola Jam Pengguna Casual vs Registered pada Hari Kerja
Menganalisis perbedaan perilaku penyewaan antara hari kerja (*working day*) dan akhir pekan/hari libur (*weekend/holiday*).


In [ ]:
# Perbandingan rata-rata penyewaan per jam pada hari kerja
working_day_hourly = hour_df[hour_df['workingday'] == 1].groupby('hr')[['casual', 'registered', 'cnt']].mean().reset_index()

print("Penyewaan pada jam sibuk pagi (07:00 - 09:00):")
print(working_day_hourly[working_day_hourly['hr'].isin([7, 8, 9])])

print()
print("Penyewaan pada jam sibuk sore (17:00 - 19:00):")
print(working_day_hourly[working_day_hourly['hr'].isin([17, 18, 19])])


In [ ]:
# Perbandingan proporsi pengguna pada Hari Kerja vs Akhir Pekan
user_prop = hour_df.groupby('workingday_label')[['casual', 'registered', 'cnt']].sum()
user_prop['casual_pct'] = (user_prop['casual'] / user_prop['cnt']) * 100
user_prop['registered_pct'] = (user_prop['registered'] / user_prop['cnt']) * 100
user_prop


*Insight Analisis Pengguna:*
- Pada hari kerja, terjadi lonjakan dramatis (*twin peaks*) pada jam **08:00 pagi** (rata-rata 477 unit/jam) dan **17:00–18:00 sore** (rata-rata 492–525 unit/jam). Lebih dari 90% pengguna di jam sibuk ini adalah **pengguna terdaftar (Registered)** yang menggunakan sepeda untuk keperluan komuter berangkat dan pulang kerja.
- Sebaliknya, pengguna **Casual** memiliki proporsi jauh lebih besar pada akhir pekan/hari libur (**28,4%**) dibandingkan pada hari kerja (**14,6%**), dengan puncak aktivitas santai di siang hari (12:00–16:00).


## 5. Teknik Analisis Lanjutan (Clustering / Binning)

*(Syarat Penilaian Bintang 5: Menerapkan teknik analisis lanjutan seperti Binning/Clustering manual tanpa ML).*

Pada analisis ini diterapkan dua teknik segmentasi berbasis aturan (*rule-based binning*):
1. **Time-of-Day Grouping (Segmentasi Waktu Harian):**
   Membagi 24 jam ke dalam 4 kelompok operasional:
   - *Dini Hari:* 00:00 – 05:00
   - *Pagi:* 06:00 – 11:00
   - *Siang/Sore:* 12:00 – 17:00
   - *Malam:* 18:00 – 23:00
   *Tujuan:* Mengetahui jendela waktu tersibuk untuk alokasi logistik dan jendela waktu tersepi untuk jadwal perbaikan (*maintenance*).

2. **Daily Demand Level Clustering (Segmentasi Tingkat Permintaan):**
   Mengelompokkan volume sewa harian (`cnt`) menggunakan pemotongan kuantil (*quantile binning*) menjadi:
   - *Low Demand* (Penyewaan Rendah)
   - *Medium Demand* (Penyewaan Sedang)
   - *High Demand* (Penyewaan Tinggi)
   *Tujuan:* Mengidentifikasi karakteristik kondisi lingkungan (suhu, kelembapan, kecepatan angin) pada hari-hari permintaan tinggi versus rendah.


In [ ]:
# 1. Segmentasi Waktu Harian (Time of Day Grouping)
def assign_time_group(hour):
    if 0 <= hour < 6:
        return 'Dini Hari (00-05)'
    elif 6 <= hour < 12:
        return 'Pagi (06-11)'
    elif 12 <= hour < 18:
        return 'Siang/Sore (12-17)'
    else:
        return 'Malam (18-23)'

hour_df['time_group'] = hour_df['hr'].apply(assign_time_group)

time_group_summary = hour_df.groupby('time_group')['cnt'].agg(
    total_volume='sum',
    avg_per_hour='mean',
    record_count='count'
).reindex(['Dini Hari (00-05)', 'Pagi (06-11)', 'Siang/Sore (12-17)', 'Malam (18-23)'])

time_group_summary


In [ ]:
# 2. Segmentasi Permintaan Harian (Demand Level Binning)
day_df['demand_level'] = pd.qcut(
    day_df['cnt'],
    q=3,
    labels=['Low Demand', 'Medium Demand', 'High Demand']
)

demand_level_summary = day_df.groupby('demand_level', observed=False).agg(
    rentals_mean=('cnt', 'mean'),
    temp_celsius_mean=('temp_celsius', 'mean'),
    hum_pct_mean=('hum_pct', 'mean'),
    windspeed_kmh_mean=('windspeed_kmh', 'mean'),
    day_count=('cnt', 'count')
)

demand_level_summary


*Insight Analisis Lanjutan:*
- Periode **Siang/Sore (12:00–17:00)** menghasilkan total penyewaan terbesar (~1,29 juta unit) dengan rata-rata 295 unit/jam, disusul periode **Malam (18:00–23:00)** dengan ~986 ribu unit. Periode **Dini Hari (00:00–05:00)** memiliki volume paling rendah (rata-rata hanya 24 unit/jam).
- Pada segmentasi tingkat permintaan, hari-hari dengan **High Demand** memiliki rata-rata temperatur hangat yang nyaman (**24,4°C**) dan kecepatan angin rendah (**11,8 km/h**). Sebaliknya, hari **Low Demand** memiliki suhu rata-rata dingin (**13,7°C**) dan angin yang lebih kencang (**14,3 km/h**).


## 6. Visualization & Explanatory Analysis

### Visualisasi 1: Pengaruh Musim dan Kondisi Cuaca terhadap Rata-rata Penyewaan Sepeda (Menjawab Pertanyaan Bisnis 1)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot Musim
season_order = ['Fall', 'Summer', 'Winter', 'Spring']
season_data = day_df.groupby('season_label')['cnt'].mean().reindex(season_order).reset_index()
colors_season = ['#1a5276' if x == season_data['cnt'].max() else '#85c1e9' for x in season_data['cnt']]

sns.barplot(data=season_data, x='season_label', y='cnt', palette=colors_season, ax=axes[0])
axes[0].set_title('Rata-rata Penyewaan Harian Berdasarkan Musim', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Musim (Season)', fontsize=12)
axes[0].set_ylabel('Rata-rata Penyewaan Harian', fontsize=12)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Anotasi angka pada batang
for p in axes[0].patches:
    height = p.get_height()
    axes[0].annotate(f'{int(height):,} unit',
                     (p.get_x() + p.get_width() / 2., height),
                     ha='center', va='bottom', fontsize=10,
                     xytext=(0, 4), textcoords='offset points')

# Plot Cuaca
weather_order = ['Clear/Partly Cloudy', 'Misty/Cloudy', 'Light Snow/Rain']
weather_data = day_df.groupby('weather_label')['cnt'].mean().reindex(weather_order).reset_index()
colors_weather = ['#1a5276' if x == weather_data['cnt'].max() else '#aed6f1' for x in weather_data['cnt']]

sns.barplot(data=weather_data, x='weather_label', y='cnt', palette=colors_weather, ax=axes[1])
axes[1].set_title('Rata-rata Penyewaan Harian Berdasarkan Kondisi Cuaca', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Kondisi Cuaca (Weather)', fontsize=12)
axes[1].set_ylabel('Rata-rata Penyewaan Harian', fontsize=12)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

for p in axes[1].patches:
    height = p.get_height()
    axes[1].annotate(f'{int(height):,} unit',
                     (p.get_x() + p.get_width() / 2., height),
                     ha='center', va='bottom', fontsize=10,
                     xytext=(0, 4), textcoords='offset points')

plt.suptitle('Analisis Pengaruh Kondisi Lingkungan Terhadap Penyewaan Sepeda (2011 - 2012)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Penjelasan Visualisasi 1:**
Grafik di atas membandingkan performa rata-rata harian penyewaan sepeda berdasarkan musim dan cuaca. 
- Pada panel kiri, **Fall (Musim Gugur)** menjadi musim paling produktif dengan rata-rata mencapai **5.644 unit/hari**, mengungguli Musim Panas (4.992 unit) dan Musim Dingin (4.728 unit), sedangkan Musim Semi paling rendah (2.604 unit).
- Pada panel kanan, tampak korelasi kuat antara kejernihan cuaca dengan permintaan: hari yang cerah mencatatkan **4.876 unit/hari**, berkurang saat berawan/berkabut (4.035 unit), dan anjlok drastis ke **1.803 unit/hari** saat terjadi hujan salju/ringan.


### Visualisasi 2: Perbandingan Pola Jam Pengguna Casual vs Registered (Menjawab Pertanyaan Bisnis 2)


In [ ]:
hourly_wd = hour_df[hour_df['workingday'] == 1].groupby('hr')[['casual', 'registered', 'cnt']].mean().reset_index()
hourly_nwd = hour_df[hour_df['workingday'] == 0].groupby('hr')[['casual', 'registered', 'cnt']].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Hari Kerja
axes[0].plot(hourly_wd['hr'], hourly_wd['registered'], color='#1f77b4', marker='o', linewidth=2.2, label='Registered (Terdaftar)')
axes[0].plot(hourly_wd['hr'], hourly_wd['casual'], color='#ff7f0e', marker='s', linewidth=2, linestyle='--', label='Casual (Santai)')
axes[0].axvspan(7, 9, color='#d5dbdb', alpha=0.5, label='Jam Sibuk Pagi (07-09)')
axes[0].axvspan(17, 19, color='#d5dbdb', alpha=0.5, label='Jam Sibuk Sore (17-19)')
axes[0].set_title('Hari Kerja (Working Day): Pola Komuter', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Jam dalam Sehari (0 - 23)', fontsize=12)
axes[0].set_ylabel('Rata-rata Penyewaan per Jam', fontsize=12)
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle=':', alpha=0.6)

# Akhir Pekan / Hari Libur
axes[1].plot(hourly_nwd['hr'], hourly_nwd['registered'], color='#2ca02c', marker='o', linewidth=2.2, label='Registered (Terdaftar)')
axes[1].plot(hourly_nwd['hr'], hourly_nwd['casual'], color='#d62728', marker='s', linewidth=2, linestyle='--', label='Casual (Santai)')
axes[1].set_title('Akhir Pekan/Libur (Weekend/Holiday): Pola Rekreasi', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Jam dalam Sehari (0 - 23)', fontsize=12)
axes[1].set_xticks(range(0, 24, 2))
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.suptitle('Pola Distribusi Jam Penyewaan: Hari Kerja vs Akhir Pekan (2011 - 2012)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Penjelasan Visualisasi 2:**
Visualisasi garis di atas memperlihatkan dikotomi perilaku pengguna yang sangat jelas:
- **Pada Hari Kerja:** Terbentuk pola kurva *bimodal* (dua puncak tajam). Puncak pertama terjadi pada jam **08:00 pagi** (rata-rata 454 unit registered vs 22 unit casual) dan puncak kedua yang lebih besar pada jam **17:00–18:00 sore** (rata-rata 468–441 unit registered vs 50–57 unit casual). Hal ini membuktikan bahwa pada hari kerja, armada sepeda mayoritas dimanfaatkan oleh pekerja komuter sebagai transportasi harian.
- **Pada Akhir Pekan/Libur:** Kurva berbentuk *unimodal* (satu bukit landai) dengan puncak aktivitas terjadi antara pukul **12:00 hingga 16:00**. Pengguna santai (*Casual*) melonjak tajam melampaui 100 unit/jam, menandakan pemakaian untuk kebutuhan rekreasi dan berolahraga.


### Visualisasi 3: Segmentasi Lanjutan (Binning Waktu Operasional & Tingkat Permintaan)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Visualisasi Binning Waktu Operasional
time_groups = ['Dini Hari (00-05)', 'Pagi (06-11)', 'Siang/Sore (12-17)', 'Malam (18-23)']
time_sums = hour_df.groupby('time_group')['cnt'].sum().reindex(time_groups)

sns.barplot(x=time_sums.index, y=time_sums.values, palette='Blues_r', ax=axes[0])
axes[0].set_title('Total Volume Penyewaan Berdasarkan Sesi Waktu', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sesi Waktu Operasional', fontsize=11)
axes[0].set_ylabel('Total Sepeda Disewa', fontsize=11)
axes[0].tick_params(axis='x', rotation=15)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height()/1000:,.1f}k',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

# Visualisasi Karakteristik Temperatur pada Kategori Permintaan
sns.boxplot(data=day_df, x='demand_level', y='temp_celsius', palette=['#f1948a', '#f8c471', '#82e0aa'], ax=axes[1])
axes[1].set_title('Distribusi Temperatur (°C) Menurut Tingkat Permintaan Harian', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tingkat Permintaan (Demand Level)', fontsize=11)
axes[1].set_ylabel('Temperatur Riil (°C)', fontsize=11)

plt.tight_layout()
plt.show()


**Penjelasan Visualisasi 3:**
- Sesi **Siang/Sore (12:00–17:00)** menyumbang pangsa pasar terbesar yaitu ~1,29 juta penyewaan, diikuti oleh sesi **Malam (18:00–23:00)** sebesar ~986 ribu penyewaan. Sesi **Dini Hari (00:00–05:00)** mencatatkan aktivitas yang sangat rendah (~106 ribu unit secara kumulatif).
- Boxplot temperatur menunjukkan korelasi positif yang signifikan: hari dengan status **High Demand** memiliki median suhu di atas 25°C, sedangkan hari **Low Demand** memiliki median suhu hanya ~12°C. Suhu hangat menjadi pendorong paling dominan dalam mendongkrak minat bersepeda.


## 7. Conclusion & Actionable Recommendation

### 7.1 Kesimpulan

1. **Kesimpulan Pertanyaan Bisnis 1 (Pengaruh Musim & Cuaca):**
   - **Musim Gugur (Fall)** adalah musim dengan rata-rata penyewaan sepeda tertinggi (**5.644 unit/hari**), disusul oleh **Musim Panas (Summer)** (**4.992 unit/hari**), dan **Musim Dingin (Winter)** (**4.728 unit/hari**). **Musim Semi (Spring)** menempati posisi terendah (**2.604 unit/hari**).
   - Kondisi cuaca cerah (**Clear**) menghasilkan penyewaan optimal (**4.876 unit/hari**). Sebaliknya, cuaca buruk/hujan (**Light Snow/Rain**) menekan penyewaan sebesar **63%** menjadi hanya **1.803 unit/hari**.

2. **Kesimpulan Pertanyaan Bisnis 2 (Pola Pengguna Jam Sibuk):**
   - Pada hari kerja (*working day*), penyewaan sepeda didominasi secara mutlak oleh **pengguna Registered (>85–90%)** dengan dua lonjakan puncak komuter (*rush hour*) pada pukul **08:00** (pagi) dan pukul **17:00–18:00** (sore).
   - Pada akhir pekan/hari libur (*weekend/holiday*), pola penggunaan bergeser ke aktivitas rekreasi siang hari (12:00–16:00) di mana kontribusi **pengguna Casual** melonjak hingga **28,4%** dari total penyewaan.

---

### 7.2 Actionable Recommendations (Rekomendasi Aksi Bisnis Nyata)

1. **Optimasi Operasional & Logistik Armada (Fleet Rebalancing):**
   - **Jadwal Rebalancing Stasiun Docking:** Tim operasional harus memfokuskan redistribusi sepeda ke kawasan pemukiman/stasiun transit sebelum pukul **07:00 pagi**, dan melakukan redistribusi balik dari kawasan perkantoran sentral sebelum pukul **16:30 sore** pada hari kerja.
   - **Jendela Pemeliharaan Terjadwal (*Maintenance Window*):** Manfaatkan sesi **Dini Hari (00:00–05:00)** dan periode **Musim Semi (Spring)** saat tingkat utilisasi sepeda berada di titik terendah untuk melakukan perbaikan berkala, pengecekan rem/rantai, dan penambahan slot docking tanpa mengganggu kepuasan pelanggan di jam sibuk.

2. **Strategi Pemasaran & Konversi Pengguna (Marketing Conversion):**
   - **Konversi Casual ke Registered:** Pada akhir pekan di musim panas dan musim gugur di mana pengguna *casual* melimpah di titik-titik wisata dan taman kota, pasang promosi *"Weekend Commuter Pass"* atau penawaran diskon konversi langganan bulanan di aplikasi mobile.

3. **Manajemen Dinamis Berbasis Cuaca (Weather-Driven Incentive):**
   - Pada hari-hari dengan prediksi cuaca buruk (hujan/suhu dingin) yang terbukti memicu *Low Demand*, berikan voucher diskon dinamis (*Rainy Day Perks*) atau integrasikan program poin loyalitas bagi anggota *registered* yang tetap aktif berkendara.
